# MSMARCO-XI — Kannada Full-Coverage Robust Kaggle Indexer

This is a new implementation inspired by the successful Bengali/Gujarati/Hindi architecture, but it removes the two failure points seen in the Kannada runs:

1. No `pip install` inside the notebook.
2. No `fsspec` remote Parquet streaming.

Source resolution is:

`Kaggle local input → Hugging Face single-file download → local Parquet`

This downloads/processes only the Kannada source file when it is not already attached, not the full multilingual 55 GB corpus.

The complete Kannada training split is indexed.

## Kaggle settings

- Internet: ON
- Accelerator: GPU
- HF_TOKEN secret: recommended
- Do not install packages from inside this notebook.

## Retrieval architecture

`All Kannada records → 384-D multilingual-E5 → FAISS IVF-PQ (if installed) OR GPU Torch fallback → candidate records → multi-strategy chunking → grounding/guardrails → small LLM`

## Kannada run specification

- Language code: `kn`
- Language: Kannada
- Training file: `kantrain.jsonl` (with Parquet support if Kaggle has a converted copy)
- Embedding: `intfloat/multilingual-e5-small`
- Embedding dimension: **384**
- Vector engine: FAISS IVF-PQ if present; otherwise GPU Torch exact fallback
- Full source-record coverage: **enabled**
- Candidate chunking: fixed-size+overlap, sentence-aware, semantic, metadata-aware

## 1. Environment — no runtime pip

In [ ]:
import sys
import importlib.util
import os
from pathlib import Path

required = [
    "numpy", "torch", "pyarrow", "transformers",
    "huggingface_hub", "tqdm"
]

missing = [name for name in required if importlib.util.find_spec(name) is None]

if missing:
    raise RuntimeError(
        "Missing required Kaggle modules: " + ", ".join(missing) +
        "\n\nUse Kaggle's built-in environment/dependency manager. "
        "This notebook intentionally does not run pip."
    )

import numpy as np
import torch
import pyarrow
import transformers
import huggingface_hub

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("Torch:", torch.__version__)
print("PyArrow:", pyarrow.__version__)
print("Transformers:", transformers.__version__)
print("CUDA:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU REQUIRED. Kaggle Settings → Accelerator → GPU, then restart."
    )

print("GPU:", torch.cuda.get_device_name(0))
torch.set_float32_matmul_precision("high")

FAISS_AVAILABLE = importlib.util.find_spec("faiss") is not None

if FAISS_AVAILABLE:
    import faiss
    print("FAISS:", getattr(faiss, "__version__", "available"))
else:
    print("FAISS not installed → GPU Torch vector-store fallback will be used.")

## 2. Hugging Face token

In [ ]:
HF_TOKEN = os.environ.get("HF_TOKEN")

try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass

print("HF token available:", bool(HF_TOKEN))

## 3. Kannada configuration

This notebook is deliberately locked to Kannada.

In [ ]:
from pathlib import Path

LANGUAGE = "kn"
LANGUAGE_NAME = "Kannada"
REPO_ID = "ai4bharat/MSMARCO-XI"
REPO_FILE_PARQUET = "train/kantrain.parquet"
MODEL_NAME = "intfloat/multilingual-e5-small"

SPLIT = "train"
FULL_RUN = True
PASSAGE_MODE = "translated"

EMBED_BATCH_SIZE = 512
MAX_LENGTH = 384
EXPECTED_DIMENSION = 384
RECORD_TEXT_MAX_CHARS = 6000

FAISS_TRAIN_RECORDS = 50_000
FAISS_NLIST = 2048
FAISS_PQ_M = 48
FAISS_PQ_BITS = 8

TORCH_BLOCK_ROWS = 32_768

BATCH_ROWS = 1024
ROWS_PER_RECORD_SHARD = 100_000

FIXED_SIZE = 500
FIXED_OVERLAP = 80
SENTENCES_PER_CHUNK = 3
SEMANTIC_THRESHOLD = 0.58

ROOT = Path("/kaggle/working/msmarco_xi_kannada")
SOURCE_ROOT = ROOT / "source"
LANG_ROOT = ROOT / LANGUAGE
RECORD_ROOT = LANG_ROOT / "records"

SOURCE_ROOT.mkdir(parents=True, exist_ok=True)
RECORD_ROOT.mkdir(parents=True, exist_ok=True)

print("Language:", LANGUAGE_NAME)
print("Source:", REPO_FILE)
print("Full run:", FULL_RUN)

## 4. Reliable Kannada source resolution

The dataset card shows Kannada as `kantrain.jsonl` / `kanval.jsonl`.
This notebook supports both the raw JSONL file and a locally attached Parquet conversion.

Order:

`Kaggle local input → local Parquet/JSONL → single-file Hugging Face download`

Only the Kannada training file is fetched when a local input is unavailable.

In [ ]:
import json
from huggingface_hub import hf_hub_download
import pyarrow.parquet as pq

REPO_ID = "ai4bharat/MSMARCO-XI"
REPO_FILE_JSONL = "train/kantrain.jsonl"
REPO_FILE_PARQUET = "train/kantrain.parquet"

def find_local_source():
    candidates = []
    for root in [Path("/kaggle/input"), Path("/kaggle/working")]:
        if not root.exists():
            continue
        try:
            for filename in ["kantrain.parquet", "kantrain.jsonl"]:
                candidates.extend(
                    p for p in root.rglob(filename)
                    if p.is_file()
                )
        except Exception:
            pass

    kaggle = [
        p for p in candidates
        if str(p).startswith("/kaggle/input/")
    ]
    return kaggle[0] if kaggle else (candidates[0] if candidates else None)

LOCAL_SOURCE = find_local_source()

SOURCE_MODE = None
PARQUET_PATH = None
JSONL_PATH = None

if LOCAL_SOURCE is not None:
    if LOCAL_SOURCE.suffix.lower() == ".parquet":
        PARQUET_PATH = Path(LOCAL_SOURCE)
        SOURCE_MODE = "KAGGLE_LOCAL_PARQUET"
    else:
        JSONL_PATH = Path(LOCAL_SOURCE)
        SOURCE_MODE = "KAGGLE_LOCAL_JSONL"
else:
    print("No local Kannada training file found.")
    print("Downloading only train/kantrain.jsonl from Hugging Face...")
    try:
        JSONL_PATH = Path(
            hf_hub_download(
                repo_id=REPO_ID,
                filename=REPO_FILE_JSONL,
                repo_type="dataset",
                revision="main",
                token=HF_TOKEN,
                local_dir=str(SOURCE_ROOT),
            )
        )
        SOURCE_MODE = "HF_SINGLE_FILE_JSONL"
    except Exception as exc:
        raise RuntimeError(
            "Kannada source unavailable. Attach kantrain.jsonl/parquet "
            "as a Kaggle Input or enable Kaggle Internet."
        ) from exc

if PARQUET_PATH is not None:
    parquet_file = pq.ParquetFile(str(PARQUET_PATH))
    SOURCE_ROWS = int(parquet_file.metadata.num_rows)
    print("Source mode:", SOURCE_MODE)
    print("Source file:", PARQUET_PATH)
    print("Source rows:", f"{SOURCE_ROWS:,}")

else:
    # Count JSONL records once, then stream the file.
    with open(JSONL_PATH, "r", encoding="utf-8") as f:
        SOURCE_ROWS = sum(1 for line in f if line.strip())

    print("Source mode:", SOURCE_MODE)
    print("Source file:", JSONL_PATH)
    print("Source rows:", f"{SOURCE_ROWS:,}")

print("Language: Kannada (kn)")
print("Full run: True")

## 5. GPU embedding model

In [ ]:
from transformers import AutoTokenizer, AutoModel

device = torch.device("cuda")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN,
)

model = AutoModel.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN,
    dtype=torch.float16,
).to(device)

model.eval()

@torch.inference_mode()
def mean_pool(hidden, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(hidden.size()).float()
    return (hidden * mask).sum(1) / torch.clamp(mask.sum(1), min=1e-9)

@torch.inference_mode()
def encode_texts(texts, prefix="passage: "):
    if not texts:
        return np.empty((0, model.config.hidden_size), dtype=np.float32)

    vectors = []

    for start in range(0, len(texts), EMBED_BATCH_SIZE):
        batch = [prefix + str(x) for x in texts[start:start + EMBED_BATCH_SIZE]]

        tokens = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors="pt",
        )
        tokens = {
            key: value.to(device, non_blocking=True)
            for key, value in tokens.items()
        }

        with torch.autocast(device_type="cuda", dtype=torch.float16):
            output = model(**tokens)
            emb = mean_pool(
                output.last_hidden_state,
                tokens["attention_mask"],
            )
            emb = torch.nn.functional.normalize(emb, p=2, dim=1)

        vectors.append(
            emb.float().cpu().numpy().astype(np.float32)
        )

    return np.vstack(vectors)

test_vector = encode_texts(["test"])
print("Embedding shape:", test_vector.shape)

if test_vector.shape[1] != EXPECTED_DIMENSION:
    raise RuntimeError(
        f"Expected {EXPECTED_DIMENSION} dimensions, got {test_vector.shape[1]}"
    )

## 6. Full-record representation

In [ ]:
def build_record_text(record):
    passages = record.get("passages") or {}
    translated = passages.get("Translated_passages") or []
    selected = passages.get("is_selected") or []

    selected_texts = [
        str(translated[i]).strip()
        for i, flag in enumerate(selected)
        if flag == 1 and i < len(translated)
        and str(translated[i]).strip()
    ]

    if not selected_texts:
        selected_texts = [
            str(text).strip()
            for text in translated[:2]
            if str(text).strip()
        ]

    pieces = [
        str(record.get("query", "")).strip(),
        str(record.get("Answer", "")).strip(),
        *selected_texts,
    ]

    return "\n".join(
        part for part in pieces if part
    )[:RECORD_TEXT_MAX_CHARS]


def normalize_record(record, local_id):
    passages = record.get("passages") or {}

    return {
        "local_id": int(local_id),
        "query_id": int(record.get("query_id", 0)),
        "query": str(record.get("query", "")),
        "answer": str(record.get("Answer", "")),
        "query_type": str(record.get("query_type", "")),
        "source_lang": str(record.get("source_lang", "")),
        "target_lang": str(record.get("target_lang", "")),
        "english_passages": [
            str(x) for x in passages.get("English_passages") or []
        ],
        "translated_passages": [
            str(x) for x in passages.get("Translated_passages") or []
        ],
        "is_selected": [
            int(x) for x in passages.get("is_selected") or []
        ],
    }

## 7. Complete record store

In [ ]:
import pyarrow as pa
import pyarrow.parquet as pq

RECORD_SCHEMA = pa.schema([
    ("local_id", pa.int64()),
    ("query_id", pa.int64()),
    ("query", pa.string()),
    ("answer", pa.string()),
    ("query_type", pa.string()),
    ("source_lang", pa.string()),
    ("target_lang", pa.string()),
    ("english_passages", pa.list_(pa.string())),
    ("translated_passages", pa.list_(pa.string())),
    ("is_selected", pa.list_(pa.int8())),
])

def write_record_shard(rows, shard_id):
    if not rows:
        return

    path = RECORD_ROOT / f"records_{shard_id:05d}.parquet"

    table = pa.Table.from_pydict(
        {
            field: [row[field] for row in rows]
            for field in RECORD_SCHEMA.names
        },
        schema=RECORD_SCHEMA,
    )

    pq.write_table(
        table,
        path,
        compression="zstd",
        compression_level=7,
        use_dictionary=True,
    )

## 8. Vector engine — FAISS when available, Torch GPU fallback otherwise

In [ ]:
import time

DIM = int(model.config.hidden_size)

INDEX_PATH = LANG_ROOT / "faiss_ivfpq.index"
VECTOR_PATH = LANG_ROOT / "record_vectors.float16.bin"
CHECKPOINT_PATH = LANG_ROOT / "checkpoint.json"
CONFIG_PATH = LANG_ROOT / "config.json"

if FAISS_AVAILABLE:
    import faiss

    train_texts = []

    for batch in parquet_file.iter_batches(
        batch_size=1024,
        columns=["query", "Answer", "passages"],
    ):
        for record in batch.to_pylist():
            text = build_record_text(record)
            if text:
                train_texts.append(text)
            if len(train_texts) >= FAISS_TRAIN_RECORDS:
                break
        if len(train_texts) >= FAISS_TRAIN_RECORDS:
            break

    train_vectors = encode_texts(
        train_texts,
        prefix="passage: ",
    )

    quantizer = faiss.IndexFlatIP(DIM)

    vector_index = faiss.IndexIVFPQ(
        quantizer,
        DIM,
        FAISS_NLIST,
        FAISS_PQ_M,
        FAISS_PQ_BITS,
        faiss.METRIC_INNER_PRODUCT,
    )

    t0 = time.perf_counter()
    vector_index.train(train_vectors)
    training_seconds = time.perf_counter() - t0

    ENGINE = "FAISS_IVFPQ"

    print("Vector engine: FAISS IVF-PQ")
    print("Training records:", len(train_texts))
    print("Training seconds:", round(training_seconds, 1))
    print("Dimension:", DIM)
    print("nlist:", FAISS_NLIST)
    print("PQ:", f"{FAISS_PQ_M}x{FAISS_PQ_BITS}")

else:
    vector_memmap = np.memmap(
        VECTOR_PATH,
        mode="w+",
        dtype=np.float16,
        shape=(SOURCE_ROWS, DIM),
    )
    vector_memmap.flush()

    vector_index = None
    training_seconds = 0.0
    ENGINE = "TORCH_GPU_EXACT_FLOAT16"

    print("Vector engine: GPU Torch exact vector store")
    print("Vector file:", VECTOR_PATH)
    print(
        "Estimated vector storage:",
        round(SOURCE_ROWS * DIM * 2 / (1024**3), 2),
        "GiB",
    )

## 8.5. Unified record iterator

In [ ]:
def iter_source_records():
    if PARQUET_PATH is not None:
        for batch in parquet_file.iter_batches(
            batch_size=BATCH_ROWS,
            columns=[
                "source_lang",
                "target_lang",
                "Answer",
                "query_id",
                "query_type",
                "passages",
                "query",
            ],
        ):
            for record in batch.to_pylist():
                yield record
    else:
        with open(JSONL_PATH, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                yield json.loads(line)

# Normalize possible key aliases found in JSONL/Parquet exports.
def normalize_input_record(record):
    if "passages" not in record and "passage" in record:
        record["passages"] = record["passage"]

    if "Answer" not in record and "answer" in record:
        record["Answer"] = record["answer"]

    return record

## 9. Full Kannada indexing with checkpoint/resume

Every source record is embedded and stored. No record is removed because `is_selected` is false.

In [ ]:
import json
from tqdm.auto import tqdm

next_id = 0

if CHECKPOINT_PATH.exists():
    checkpoint = json.loads(
        CHECKPOINT_PATH.read_text(encoding="utf-8")
    )
    next_id = int(checkpoint.get("next_id", 0))

    if ENGINE == "FAISS_IVFPQ" and INDEX_PATH.exists():
        vector_index = faiss.read_index(str(INDEX_PATH))

    print("Resume point:", next_id)

pending_texts = []
pending_ids = []
pending_records = []

shard_rows = []
shard_id = next_id // ROWS_PER_RECORD_SHARD

index_start = time.perf_counter()

def checkpoint():
    CHECKPOINT_PATH.write_text(
        json.dumps(
            {
                "next_id": int(next_id),
                "vectors_indexed": int(next_id),
                "source_rows": int(SOURCE_ROWS),
                "engine": ENGINE,
            },
            indent=2,
        ),
        encoding="utf-8",
    )


def flush_pending():
    global pending_texts, pending_ids, pending_records
    global shard_rows, shard_id

    if not pending_texts:
        return

    vectors = encode_texts(
        pending_texts,
        prefix="passage: ",
    )

    ids = np.asarray(
        pending_ids,
        dtype=np.int64,
    )

    if ENGINE == "FAISS_IVFPQ":
        vector_index.add_with_ids(
            vectors,
            ids,
        )
    else:
        vector_memmap[
            ids[0]:ids[-1] + 1
        ] = vectors.astype(np.float16)

    shard_rows.extend(pending_records)

    while len(shard_rows) >= ROWS_PER_RECORD_SHARD:
        current = shard_rows[:ROWS_PER_RECORD_SHARD]
        del shard_rows[:ROWS_PER_RECORD_SHARD]

        write_record_shard(
            current,
            shard_id,
        )
        shard_id += 1

    pending_texts.clear()
    pending_ids.clear()
    pending_records.clear()

    if ENGINE == "FAISS_IVFPQ":
        faiss.write_index(
            vector_index,
            str(INDEX_PATH),
        )
    else:
        vector_memmap.flush()

    checkpoint()


for batch in tqdm(
    parquet_file.iter_batches(
        batch_size=BATCH_ROWS,
        columns=[
            "source_lang",
            "target_lang",
            "Answer",
            "query_id",
            "query_type",
            "passages",
            "query",
        ],
    ),
    desc="FULL ASSAMESE INDEX",
):
    for record in batch.to_pylist():

        pending_texts.append(
            build_record_text(record)
        )
        pending_ids.append(next_id)
        pending_records.append(
            normalize_record(
                record,
                next_id,
            )
        )

        next_id += 1

        if len(pending_texts) >= EMBED_BATCH_SIZE:
            flush_pending()

flush_pending()

if shard_rows:
    write_record_shard(
        shard_rows,
        shard_id,
    )

if ENGINE == "FAISS_IVFPQ":
    faiss.write_index(
        vector_index,
        str(INDEX_PATH),
    )
else:
    vector_memmap.flush()

elapsed_seconds = time.perf_counter() - index_start

config = {
    "dataset": REPO_ID,
    "language": LANGUAGE,
    "language_name": LANGUAGE_NAME,
    "source_file": REPO_FILE,
    "source_mode": SOURCE_MODE,
    "source_rows": int(SOURCE_ROWS),
    "records_indexed": int(next_id),
    "all_records": True,
    "all_passages_preserved": True,
    "embedding_model": MODEL_NAME,
    "embedding_dimension": DIM,
    "vector_engine": ENGINE,
    "index_build_seconds": elapsed_seconds,
    "chunking": {
        "strategies": [
            "fixed_size_overlap",
            "sentence_aware",
            "semantic",
            "metadata_aware",
        ],
        "fixed_size": FIXED_SIZE,
        "fixed_overlap": FIXED_OVERLAP,
        "sentences_per_chunk": SENTENCES_PER_CHUNK,
        "semantic_threshold": SEMANTIC_THRESHOLD,
    },
}

if ENGINE == "FAISS_IVFPQ":
    config.update({
        "index_type": "IVFPQ",
        "nlist": FAISS_NLIST,
        "pq_m": FAISS_PQ_M,
        "pq_bits": FAISS_PQ_BITS,
    })
else:
    config.update({
        "index_type": "exact_blocked_gpu_cosine",
        "vector_file": str(VECTOR_PATH),
        "vector_dtype": "float16",
    })

CONFIG_PATH.write_text(
    json.dumps(
        config,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

print("\nFULL ASSAMESE INDEX COMPLETE")
print("Source rows:", f"{SOURCE_ROWS:,}")
print("Vectors:", f"{next_id:,}")
print("Engine:", ENGINE)
print("Elapsed:", round(elapsed_seconds, 1), "seconds")

## 10. Multi-strategy candidate chunking

In [ ]:
import re

def fixed_chunks(text, size=FIXED_SIZE, overlap=FIXED_OVERLAP):
    text = str(text).strip()
    if not text:
        return []

    chunks = []
    start = 0

    while start < len(text):
        end = min(start + size, len(text))
        piece = text[start:end].strip()
        if piece:
            chunks.append(piece)

        if end >= len(text):
            break

        start += size - overlap

    return chunks


def sentence_chunks(text):
    sentences = [
        s.strip()
        for s in re.split(
            r"(?<=[.!?।॥])\s+",
            str(text).strip(),
        )
        if s.strip()
    ]

    return [
        " ".join(
            sentences[i:i + SENTENCES_PER_CHUNK]
        )
        for i in range(
            0,
            len(sentences),
            SENTENCES_PER_CHUNK,
        )
    ]


def metadata_aware_chunks(text, query_type, language):
    return [
        f"[type={query_type} language={language}] {chunk}"
        for chunk in sentence_chunks(text)
    ]


def semantic_chunks(text):
    sentences = [
        s.strip()
        for s in re.split(
            r"(?<=[.!?।॥])\s+",
            str(text).strip(),
        )
        if s.strip()
    ]

    if len(sentences) <= 1:
        return sentences

    vectors = encode_texts(
        sentences,
        prefix="passage: ",
    )

    chunks = []
    current = [sentences[0]]

    for i in range(1, len(sentences)):
        similarity = float(
            np.dot(
                vectors[i - 1],
                vectors[i],
            )
        )

        if similarity < SEMANTIC_THRESHOLD:
            chunks.append(" ".join(current))
            current = [sentences[i]]
        else:
            current.append(sentences[i])

    if current:
        chunks.append(" ".join(current))

    return chunks

print(
    "Chunking ready:",
    "fixed+overlap | sentence | semantic | metadata-aware"
)

## 11. Retrieval benchmark harness

100 source queries are used for the retrieval benchmark.

These P50/P70/P100 values are **retrieval-stage values**. The final application must measure the complete voice-to-answer pipeline separately.

In [ ]:
def retrieve(query, top_k=20):

    query_vector = encode_texts(
        [query],
        prefix="query: ",
    )[0]

    if ENGINE == "FAISS_IVFPQ":

        scores, ids = vector_index.search(
            query_vector.reshape(1, -1),
            top_k,
        )

        return [
            (int(idx), float(score))
            for idx, score in zip(
                ids[0],
                scores[0],
            )
            if idx >= 0
        ]

    # Exact GPU fallback.
    q = torch.from_numpy(
        query_vector
    ).to(
        device,
        dtype=torch.float16,
    )

    best_scores = torch.empty(
        0,
        device=device,
        dtype=torch.float16,
    )
    best_ids = torch.empty(
        0,
        device=device,
        dtype=torch.long,
    )

    for start in range(
        0,
        int(SOURCE_ROWS),
        TORCH_BLOCK_ROWS,
    ):
        end = min(
            start + TORCH_BLOCK_ROWS,
            int(SOURCE_ROWS),
        )

        block = torch.from_numpy(
            np.asarray(
                vector_memmap[start:end]
            )
        ).to(
            device,
            dtype=torch.float16,
        )

        scores = torch.matmul(
            block,
            q,
        )

        values, indices = torch.topk(
            scores,
            min(top_k, scores.shape[0]),
        )

        indices = indices + start

        merged_scores = torch.cat(
            [best_scores, values]
        )
        merged_ids = torch.cat(
            [best_ids, indices]
        )

        k = min(
            top_k,
            merged_scores.shape[0],
        )

        best_scores, positions = torch.topk(
            merged_scores,
            k,
        )
        best_ids = merged_ids[positions]

    return [
        (int(idx), float(score))
        for idx, score in zip(
            best_ids.cpu().tolist(),
            best_scores.cpu().tolist(),
        )
    ]

In [ ]:
benchmark_queries = []

for raw_record in iter_source_records():
    record = normalize_input_record(raw_record)
    q = str(record.get("query", "")).strip()
    if q:
        benchmark_queries.append(q)
    if len(benchmark_queries) >= 100:
        break

latencies_ms = []

for query in benchmark_queries:
    start = time.perf_counter()

    retrieve(
        query,
        top_k=20,
    )

    latencies_ms.append(
        (time.perf_counter() - start) * 1000
    )

latencies_ms = np.asarray(
    latencies_ms,
    dtype=np.float64,
)

print("Queries:", len(latencies_ms))
print(f"P50: {np.percentile(latencies_ms, 50):.2f} ms")
print(f"P70: {np.percentile(latencies_ms, 70):.2f} ms")
print(f"P100: {np.max(latencies_ms):.2f} ms")

## 12. Production harness + guardrail specification

Final online flow:

```text
STT
 ↓
language detection
 ↓
query validation
 ↓
query embedding
 ↓
vector retrieval
 ↓
candidate passage extraction
 ↓
one of: sentence / fixed / semantic / metadata-aware
 ↓
reranking
 ↓
grounding gate
 ↓
small LLM
 ↓
answer validation
 ↓
final response
```

Grounding rule:

`No sufficiently similar retrieved context → do not answer from model memory.`

Structured result:

```json
{
  "answer": "...",
  "language": "as",
  "retrieved_ids": [],
  "chunk_strategy": "sentence",
  "grounded": true,
  "latency_ms": {
    "stt": 0,
    "embedding": 0,
    "retrieval": 0,
    "chunking": 0,
    "guardrail": 0,
    "llm": 0,
    "total": 0
  }
}
```

Use retries only for transient service failures; do not retry deterministic guardrail failures.

## 13. Final artifacts

If FAISS exists:

```text
as/
├── faiss_ivfpq.index
├── config.json
├── checkpoint.json
└── records/
```

If FAISS is absent:

```text
as/
├── record_vectors.float16.bin
├── config.json
├── checkpoint.json
└── records/
```

`records/*.parquet` contains the source records and all passages.

`config.json` records the exact vector engine, embedding dimension, source row count, and chunking configuration.

`checkpoint.json` exists only to resume an interrupted full build.